# Phase 1D.3 Colab run

Use a fresh Tesla T4 runtime. This notebook stops on environment or wheel provenance mismatches. It does not create a public tunnel. Save cell outputs as evidence.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tarfile
archive=os.getenv('PHASE1D3_SOURCE_ARCHIVE')
expected=os.getenv('PHASE1D3_SOURCE_SHA256')
if archive:
    archive_path=Path(archive)
    if not expected or hashlib.sha256(archive_path.read_bytes()).hexdigest()!=expected:
        raise RuntimeError('BLOCKED: source archive SHA-256 is missing or mismatched.')
    extract_root=Path('/content/phase-1d3-source-verified')
    if extract_root.exists(): raise RuntimeError('Use a fresh extraction path for this reviewed source archive.')
    extract_root.mkdir()
    with tarfile.open(archive_path,'r:*') as bundle:
        bundle.extractall(extract_root, filter='data')
    candidates=[extract_root]+[path for path in extract_root.iterdir() if path.is_dir()]
    REPO=next((path for path in candidates if (path/'ai/runtime/app.py').is_file()),None)
    if REPO is None: raise RuntimeError('BLOCKED: verified archive does not contain the runtime source.')
    source_proof={'commit':None,'archive':str(archive_path),'sha256':expected}
else:
    REPO=Path('/content/phase-1d3-repo')
    commit=os.getenv('PHASE1D3_REVIEWED_COMMIT')
    if not (REPO/'.git').exists() or not commit: raise RuntimeError('BLOCKED: provide a reviewed clean Git checkout or a SHA-verified tar archive.')
    actual=subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
    dirty=subprocess.run(['git','status','--porcelain','--untracked-files=all'],cwd=REPO,capture_output=True,text=True,check=True).stdout
    if actual!=commit or dirty: raise RuntimeError('BLOCKED: source commit mismatch or checkout is dirty.')
    source_proof={'commit':actual,'archive':None,'sha256':None}
os.chdir(REPO)
if not (REPO/'ai/runtime/app.py').is_file(): raise RuntimeError('Reviewed source checkout is incomplete.')
Path('/content/dacn-source-proof.json').write_text(json.dumps(source_proof,indent=2)+'\n')
print('verified source:',source_proof)

In [ ]:
# Snapshot before any dependency operation. Never install/replace Torch.
import json, platform, torch, hashlib
snapshot = {'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.version.cuda,
 'pipFreeze':subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True).stdout.splitlines(),
 'torchFiles':{}, 'gpu':subprocess.run(['nvidia-smi','--query-gpu=name,uuid,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()}

for rel in __import__('importlib.metadata').metadata.distribution('torch').files:
    path=Path(torch.__file__).parent.parent / rel
    if path.is_file(): snapshot['torchFiles'][str(rel)]=hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(snapshot,indent=2))
if not snapshot['python'].startswith('3.13.') or snapshot['torch']!='2.11.0+cu130' or snapshot['cuda']!='13.0' or 'Tesla T4' not in snapshot['gpu']:
    raise RuntimeError('BLOCKED: Python/Torch/CUDA/T4 candidate mismatch; stop without changing Torch.')
Path('/content/dacn-colab-before.json').write_text(json.dumps(snapshot,indent=2)+'\n')

## Dependency lock gate

Establish the exact installed Torch wheel URL and SHA-256 from an official source and compare wheel RECORD hashes with installed files. If that cannot be proven, stop and return the snapshot. Never guess a PyTorch index. In a separate Python 3.13/Linux resolver environment, run python -m pip install pip-tools==7.5.2, then pip-compile --generate-hashes --output-file ai/requirements.lock --constraint ai/constraints-colab.txt ai/requirements.in with the exact already-observed official PyTorch wheel index configured for that resolver. Do not guess the index. Preserve the installed Torch wheel URL and SHA-256, compare its RECORD to every installed Torch file, then make /content/torch-wheel-provenance.json with version, wheelPath, officialUrl and sha256. Copy the complete lock into the reviewed source checkout. The repository lock stays Pending until the graph, hashes and no-Torch-action dry run are verified.

In [ ]:
from ai.colab.preflight import validate_install_plan, verify_torch_wheel
subprocess.run([sys.executable,'-m','venv','--system-site-packages','/content/dacn-venv'],check=True)
VENV_PY='/content/dacn-venv/bin/python'
node_version=subprocess.run(['node','--version'],capture_output=True,text=True,check=True).stdout.strip().lstrip('v')
if not node_version.startswith('24.') or tuple(map(int,node_version.split('.')[:2]))<(24,18): raise RuntimeError('Node must be >=24.18.0 and <25.')
subprocess.run(['corepack','enable'],check=True)
if subprocess.run(['pnpm','--version'],capture_output=True,text=True,check=True).stdout.strip()!='10.12.1': raise RuntimeError('pnpm must be 10.12.1.')
subprocess.run(['pnpm','install','--frozen-lockfile'],cwd=REPO,check=True)
lock=REPO/'ai/requirements.lock'; proof=Path('/content/torch-wheel-provenance.json'); plan=Path('/content/dacn-install-plan.json')
if not lock.is_file() or not proof.is_file():
    raise RuntimeError('BLOCKED: complete reviewed lock, Torch provenance, and pip dry-run report are required; nothing installed.')
provenance=json.loads(proof.read_text())
if provenance.get('version')!='2.11.0+cu130' or not verify_torch_wheel(provenance.get('wheelPath',''),provenance.get('officialUrl',''),provenance.get('sha256',''),snapshot['torchFiles']):
    raise RuntimeError('BLOCKED: official Torch wheel bytes or installed RECORD correspondence could not be verified.')
subprocess.run([VENV_PY,'-m','pip','install','--dry-run','--report',str(plan),'-r',str(lock),'-c',str(REPO/'ai/constraints-colab.txt')],check=True)
if not validate_install_plan(json.loads(plan.read_text())):
    raise RuntimeError('BLOCKED: pip plan would install/replace Torch.')
subprocess.run([VENV_PY,'-m','pip','install','--require-hashes','-r',str(lock),'-c',str(REPO/'ai/constraints-colab.txt')],check=True)
subprocess.run([VENV_PY,'-m','pip','check'],check=True)
subprocess.run([VENV_PY,'-c','import fastapi,uvicorn,jsonschema,transformers,accelerate,bitsandbytes'],check=True)
verify="import torch,json,hashlib; from pathlib import Path; d=__import__('importlib.metadata').metadata.distribution('torch'); root=Path(torch.__file__).parent.parent; files={str(r):hashlib.sha256((root/r).read_bytes()).hexdigest() for r in d.files if (root/r).is_file()}; print(json.dumps({'version':torch.__version__,'cuda':torch.version.cuda,'files':files}))"
verified=json.loads(subprocess.run([VENV_PY,'-c',verify],capture_output=True,text=True,check=True).stdout)
if verified['version']!=snapshot['torch'] or verified['cuda']!=snapshot['cuda'] or verified['files']!=snapshot['torchFiles']:
    raise RuntimeError('BLOCKED: venv Torch/CUDA or installed files differ from snapshot.')
torch_after=__import__('torch')
if torch_after.__version__!=snapshot['torch'] or torch_after.version.cuda!=snapshot['cuda']:
    raise RuntimeError('BLOCKED: Torch/CUDA changed during dependency install.')
for rel,digest in snapshot['torchFiles'].items():
    path=Path(torch_after.__file__).parent.parent / rel
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest()!=digest:
        raise RuntimeError('BLOCKED: installed Torch files changed.')

In [ ]:
# Partial preflight: modelLoad stays false until the real worker starts under the GPU lock.
from ai.colab.preflight import preflight_environment
probe="import json,torch,transformers,bitsandbytes; cfg=transformers.BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16); print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'nf4Import':isinstance(cfg,transformers.BitsAndBytesConfig)}))"
probe_result=json.loads(subprocess.run([VENV_PY,'-c',probe],capture_output=True,text=True,check=True).stdout)
checks={'python':snapshot['python'],'torch':probe_result['torch'],'cuda':probe_result['cuda'],'gpu':'Tesla T4',
        'pipCheck':True,'nf4Import':probe_result['nf4Import'],'modelLoad':False}
print(json.dumps(preflight_environment(checks),indent=2))

## Run benchmark

Keep Uvicorn local. Supply the bearer token through a hidden prompt. Do not create a tunnel here; establish authorized private reachability separately.

In [ ]:
from getpass import getpass
import hashlib, time
from ai.colab.preflight import preflight_environment
os.environ['AI_RUNTIME_TOKEN']=getpass('Runtime bearer token (hidden): ')
os.environ['AI_RUNTIME_MODEL_ID']='Qwen/Qwen3-4B-Instruct-2507'
os.environ['AI_RUNTIME_MODEL_REVISION']='cdbee75f17c01a7cc42f958dc650907174af0554'
os.environ['AI_RUNTIME_GPU_UUID']=snapshot['gpu'].split(',')[1].strip()
os.environ['AI_BENCHMARK_GPU']=snapshot['gpu'].split(',')[0].strip()
os.environ['AI_BENCHMARK_GPU_UUID']=snapshot['gpu'].split(',')[1].strip()
os.environ['AI_BENCHMARK_TORCH']=snapshot['torch']
os.environ['AI_BENCHMARK_CUDA']=snapshot['cuda']
from ai.benchmark.allowlist import fixture_request_hash
fixture_rows=[json.loads(line) for line in (REPO/'ai/benchmark/fixtures/fixtures.jsonl').read_text().splitlines()]
os.environ['AI_RUNTIME_BENCHMARK_MODE']='1'
os.environ['AI_RUNTIME_BENCHMARK_ALLOWLIST']=','.join(sorted({fixture_request_hash(row['task'],row['input']) for row in fixture_rows}))
os.environ['AI_RUNTIME_OBSERVATION_DIR']='/content/dacn-raw-observations'
node_version=subprocess.run(['node','--version'],capture_output=True,text=True,check=True).stdout.strip().lstrip('v')
if not node_version.startswith('24.') or tuple(map(int,node_version.split('.')[:2]))<(24,18): raise RuntimeError('Node must be >=24.18.0 and <25.')
pnpm_version=subprocess.run(['pnpm','--version'],capture_output=True,text=True,check=True).stdout.strip()
if pnpm_version!='10.12.1': raise RuntimeError('pnpm must be 10.12.1.')
server=subprocess.Popen([VENV_PY,'-m','uvicorn','ai.runtime.app:app','--host','127.0.0.1','--port','8000'],cwd=REPO,stdout=open('/content/dacn-runtime.log','w'),stderr=subprocess.STDOUT)
for _ in range(600):
    if server.poll() is not None: raise RuntimeError('Runtime/model startup failed; inspect local log.')
    log=Path('/content/dacn-runtime.log').read_text() if Path('/content/dacn-runtime.log').exists() else ''
    if 'Application startup complete' in log: break
    time.sleep(1)
else: server.terminate(); raise RuntimeError('Runtime/model startup timed out.')
checks['modelLoad']=True
result=preflight_environment(checks); print(json.dumps(result,indent=2))
if result['status']!='READY': server.terminate(); raise RuntimeError('BLOCKED: model/runtime preflight failed.')
subprocess.run([VENV_PY,'ai/colab/measure_capacity.py'],cwd=REPO,check=True)
subprocess.run([VENV_PY,'ai/benchmark/validate_fixtures.py'],cwd=REPO,check=True)
os.environ['AI_BENCHMARK_MODE']='1'
from ai.runtime.prompts import PROMPT_VERSIONS
os.environ['AI_BENCHMARK_CONFIG_HASH']=hashlib.sha256(json.dumps({'model':'Qwen/Qwen3-4B-Instruct-2507','revision':'cdbee75f17c01a7cc42f958dc650907174af0554','torch':snapshot['torch'],'cuda':snapshot['cuda'],'promptVersions':PROMPT_VERSIONS},sort_keys=True).encode()).hexdigest()
run_result=subprocess.run([VENV_PY,'ai/benchmark/run.py','--fixtures','ai/benchmark/fixtures/fixtures.jsonl','--endpoint','http://127.0.0.1:8000/v1/infer','--output-dir','/content/dacn-benchmark','--export-output','/content/dacn-benchmark/outputs'],cwd=REPO,capture_output=True,text=True,check=True)
run_dir=Path(json.loads(run_result.stdout)['runDir'])
print(f'Benchmark run directory: {run_dir}')

In [ ]:
# After manually adding scores.json, run this cell. It remains Pending until verified evidence.json is also present.
if 'run_dir' not in globals() or not (run_dir/'manifest.json').exists(): raise RuntimeError('Run the benchmark cell first; set run_dir to the intended resumed run when resuming.')
subprocess.run([VENV_PY,'ai/benchmark/report.py','--run-dir',str(run_dir)],cwd=REPO,check=True)
# Add one five-dimension score object per fixture to run_dir/scores.json before reporting final PASS eligibility.
import hashlib
artifact_roots=[Path('/content/dacn-benchmark'),Path('/content/dacn-source-proof.json'),Path(REPO/'ai/requirements.lock'),Path('/content/dacn-raw-observations'),Path('/content/dacn-capacity-measurements.json'),Path('/content/dacn-colab-before.json'),Path('/content/dacn-install-plan.json'),Path('/content/torch-wheel-provenance.json'),Path('/content/dacn-runtime.log')]
manifest={}
for item in artifact_roots:
    files=[item] if item.is_file() else ([f for f in item.rglob('*') if f.is_file()] if item.exists() else [])
    for file in files:
        if file.name!='SHA256SUMS.json': manifest[str(file)]=hashlib.sha256(file.read_bytes()).hexdigest()
Path('/content/dacn-phase1d3-SHA256SUMS.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('Evidence hash manifest: /content/dacn-phase1d3-SHA256SUMS.json')
import zipfile
archive='/content/dacn-phase1d3-evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
    for item in artifact_roots+[Path('/content/dacn-nestjs-transport-evidence.json')]:
        files=[item] if item.is_file() else ([f for f in item.rglob('*') if f.is_file()] if item.exists() else [])
        for file in files:
            if file.name!='SHA256SUMS.json': bundle.write(file,arcname=str(file).lstrip('/'))
    bundle.write('/content/dacn-phase1d3-SHA256SUMS.json',arcname='dacn-phase1d3-SHA256SUMS.json')
print('Return this archive:',archive)


## Return evidence

Review every output with ai/benchmark/rubric.md and save all five scores per attempted fixture. Return the full run directory, source hash, lock and wheel evidence, environment snapshot, fixture hash, attempts, evaluator results, scores, capacity measurements and NestJS transport evidence. Do not activate a profile or close OPEN-ARCH-003.